# 세션 기억과 장기 기억 Store — 실습본

원본: `src/raw/Chapter02/03-langgraph-advanced/03_memory_store.py`  
슬라이드: `docs/raw/Chapter02/day01/3-4-langgraph-advanced.pdf`

## 학습 목표

- Checkpointer와 Store의 범위·namespace
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

메시지 → Store 검색 → 기억 포함 프롬프트 → LLM → 사실 저장

**읽을 때 확인할 점:** 실습본 compile()에는 checkpointer와 store가 빠져 있고 정답본에는 있다. 실습본의 세션 간 기억 시연은 Store 조회 결과와 비교한다.


## 실행 전 확인

LLM 호출과 메모리 Store 쓰기가 일어난다. Store는 커널을 재시작하면 사라진다.

아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root / "src"))
__file__ = str(_repo_root / "src/raw/Chapter02/03-langgraph-advanced/03_memory_store.py")


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
import re
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.store.memory import InMemoryStore
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


# ── Checkpointer vs Store ─────────────────────────────────────────────────────
#
# Checkpointer (MemorySaver 등)
#   - thread(세션) 단위로 State를 저장
#   - 같은 thread 안에서 대화 흐름을 이어감 (단기 기억)
#
# Store (InMemoryStore 등)
#   - thread를 넘어 여러 세션에서 공유되는 장기 기억
#   - 사용자 프로필, 선호도, 중요 사실 등을 저장
#   - namespace (tuple)로 데이터를 논리적으로 분리한다

# TODO: 장기 기억을 저장할 Store를 생성해 주세요.
store = InMemoryStore()


class State(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def chatbot(state: State) -> dict:
    user_id = state.get('user_id', 'anonymous')

    # Store에서 이 사용자의 장기 기억 조회
    namespace = ('memories', user_id)
    # TODO: Store에서 현재 사용자의 장기 기억을 조회해 주세요.
    memories = store.search(namespace)
    memory_text = '\n'.join(f'- {m.value["fact"]}' for m in memories) if memories else '없음'

    system_prompt = f"""너는 친절한 AI 어시스턴트야.

사용자에 대해 알고 있는 정보:
{memory_text}

대화 중 사용자의 중요한 정보(이름, 취미, 직업 등)가 나오면,
"[기억 저장: <내용>]" 형식으로 메시지 앞에 표시해줘."""

    response = llm.invoke([SystemMessage(content=system_prompt)] + state['messages'])

    # LLM이 기억할 내용을 표시했으면 Store에 저장
    content = response.content
    if '[기억 저장:' in content:
        facts = re.findall(r'\[기억 저장: (.+?)\]', content)
        for i, fact in enumerate(facts):
            key = f'fact_{len(memories) + i}'
            # TODO: 추출한 fact를 Store에 저장해 주세요.
            store.put(namespace, key, {'fact': fact})

    return {'messages': [response]}


builder = StateGraph(State)
# TODO: chatbot 노드와 START/END 엣지를 연결해 주세요.
builder.add_node('chatbot', chatbot)
builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)
# TODO: MemorySaver 체크포인터와 Store를 사용해 그래프를 컴파일해 주세요.
app = builder.compile()


def chat(user_id: str, thread_id: str, message: str) -> str:
    config = {'configurable': {'thread_id': thread_id}}
    result = app.invoke(
        {'messages': [HumanMessage(content=message)], 'user_id': user_id},
        config=config,
    )
    return result['messages'][-1].content


print('=== 세션 1: 정보 입력 ===')
print(chat('user_001', 'session_1', '안녕! 나는 이지수야. 취미는 피아노 연주야.'))

print('\n=== 세션 2: 새 세션에서도 기억 유지 ===')
print(chat('user_001', 'session_2', '내 이름이랑 취미 알아?'))

print('\n=== Store에 저장된 장기 기억 ===')
memories = store.search(('memories', 'user_001'))
for m in memories:
    print(f'  [{m.key}] {m.value["fact"]}')


## CodeMate와 함께 검토

> 실습본과 정답본의 compile을 비교하고 기억 저장 위치와 정규식 기반 사실 추출의 한계를 설명해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. 사용자별 기억 구분 값은?
2. 모델이 [기억 저장: ...]을 출력하지 않으면?
